In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from ml_gearbox.audit import (
    schema_report, target_report, duplicate_report, missingness_report,
    hidden_missing_report, category_normalization_check, infinite_value_report,
    cardinality_report, constant_feature_report, numeric_profile, range_validation
)

In [4]:
DATA_PATH = Path("../data/raw/data.csv")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "loan_status"

In [5]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (32581, 12)


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [6]:
display(schema_report(df))
display(target_report(df, TARGET))
display(missingness_report(df))
display(hidden_missing_report(df))
display(infinite_value_report(df))
print("Duplicates:", duplicate_report(df))

,dtype,missing_count,missing_pct,n_unique,unique_pct
loan_int_rate,float64,3116,9.563856,349,1.071176
person_emp_length,float64,895,2.747000,37,0.113563
person_income,int64,0,0.000000,4295,13.182530
person_age,int64,0,0.000000,58,0.178018
person_home_ownership,str,0,0.000000,4,0.012277
loan_intent,str,0,0.000000,6,0.018416
loan_grade,str,0,0.000000,7,0.021485
loan_amnt,int64,0,0.000000,753,2.311163
loan_status,int64,0,0.000000,2,0.006139
loan_percent_income,float64,0,0.000000,77,0.236334


,class,count,percentage
0,0,25473,78.183604
1,1,7108,21.816396


,feature,missing_count,percentage
0,loan_int_rate,3116,9.563856
1,person_emp_length,895,2.747000


,feature,hidden_missing_count,percentage


,feature,infinite_count


Duplicates: {'full_row_duplicates': 165}


In [7]:
# categorical columns data audit
cat_cols = df.select_dtypes(include=["object", "category", "string"]).columns.tolist()

display(category_normalization_check(df, cat_cols))
display(cardinality_report(df))
display(constant_feature_report(df))
display(numeric_profile(df))

,feature,original_unique,normalized_unique


,feature,n_unique
0,loan_grade,7
1,loan_intent,6
2,person_home_ownership,4
3,cb_person_default_on_file,2


,feature,dominant_percentage,is_strictly_constant
0,cb_person_default_on_file,82.367024,False
1,loan_status,78.183604,False
2,person_home_ownership,50.477272,False
3,loan_grade,33.077561,False
4,loan_intent,19.806022,False
5,cb_person_cred_hist_length,18.308216,False
6,person_emp_length,12.599368,False
7,person_age,11.936405,False
8,loan_int_rate,9.563856,False
9,loan_amnt,8.176545,False


,count,mean,std,min,25%,50%,75%,max,median,skew,n_unique
person_income,32581.0,66074.848470,61983.119168,4000.00,38500.00,55000.00,79200.00,6000000.00,55000.00,32.865349,4295
person_emp_length,31686.0,4.789686,4.142630,0.00,2.00,4.00,7.00,123.00,4.00,2.614455,36
person_age,32581.0,27.734600,6.348078,20.00,23.00,26.00,30.00,144.00,26.00,2.581393,58
cb_person_cred_hist_length,32581.0,5.804211,4.055001,2.00,3.00,4.00,8.00,30.00,4.00,1.661790,29
loan_status,32581.0,0.218164,0.413006,0.00,0.00,0.00,0.00,1.00,0.00,1.364888,2
loan_amnt,32581.0,9589.371106,6322.086646,500.00,5000.00,8000.00,12200.00,35000.00,8000.00,1.192477,753
loan_percent_income,32581.0,0.170203,0.106782,0.00,0.09,0.15,0.23,0.83,0.15,1.064669,77
loan_int_rate,29465.0,11.011695,3.240459,5.42,7.90,10.99,13.47,23.22,10.99,0.208550,348


In [8]:
#rules for the dataset to be inorder to be considered valid for training..

rules = {
    "person_age": {"min": 18, "max": 110},
    "person_income": {"min": 0},
    "person_emp_length": {"min": 0},
    "loan_amnt": {"min": 1},
    "loan_int_rate": {"min": 0, "max": 100},
    "loan_percent_income": {"min": 0},
    "cb_person_cred_hist_length": {"min": 0},
    "loan_status": {"allowed": [0, 1]},
}


In [9]:
display(range_validation(df, rules))
print("Age/employment suspicious rows:", ((df["person_emp_length"] > df["person_age"] - 14) & df["person_emp_length"].notna()).sum())

valid = df["person_income"] > 0
ratio_err = (df.loc[valid, "loan_amnt"] / df.loc[valid, "person_income"] - df.loc[valid, "loan_percent_income"]).abs()
print("Ratio mismatches (>0.02):", (ratio_err > 0.02).sum())
print("Zero-income rows:", (~valid).sum())
print("Full identical rows:", df.duplicated().sum())

,feature,check,violation_count,violation_pct
0,person_age,min >= 18,0,0.00
1,person_age,max <= 110,5,0.02
2,person_income,min >= 0,0,0.00
3,person_emp_length,min >= 0,0,0.00
4,loan_amnt,min >= 1,0,0.00
5,loan_int_rate,min >= 0,0,0.00
6,loan_int_rate,max <= 100,0,0.00
7,loan_percent_income,min >= 0,0,0.00
8,cb_person_cred_hist_length,min >= 0,0,0.00
9,loan_status,allowed_values,0,0.00


Age/employment suspicious rows: 2
Ratio mismatches (>0.02): 283
Zero-income rows: 0
Full identical rows: 165


In [10]:
from sklearn.model_selection import train_test_split

In [11]:
train_df, test_df = train_test_split(
    df, test_size=0.20, stratify=df[TARGET], random_state=42
)

In [12]:
train_df = train_df.copy()
test_df = test_df.copy()

train_df.to_parquet(OUTPUT_DIR / "train.parquet", index=False)
test_df.to_parquet(OUTPUT_DIR / "test.parquet", index=False)

print("Train:", train_df.shape, "Test:", test_df.shape)
print("Train class proportions:", train_df[TARGET].value_counts(normalize=True))
print("Test class proportions:", test_df[TARGET].value_counts(normalize=True))

Train: (26064, 12) Test: (6517, 12)
Train class proportions: loan_status
0    0.781845
1    0.218155
Name: proportion, dtype: float64
Test class proportions: loan_status
0    0.781801
1    0.218199
Name: proportion, dtype: float64


In [13]:
from ml_gearbox.audit import run_data_audit

In [15]:
print(run_data_audit(df, "loan_status"))

{'schema':                               dtype  missing_count  missing_pct  n_unique  \
loan_int_rate               float64           3116     9.563856       349   
person_emp_length           float64            895     2.747000        37   
person_income                 int64              0     0.000000      4295   
person_age                    int64              0     0.000000        58   
person_home_ownership           str              0     0.000000         4   
loan_intent                     str              0     0.000000         6   
loan_grade                      str              0     0.000000         7   
loan_amnt                     int64              0     0.000000       753   
loan_status                   int64              0     0.000000         2   
loan_percent_income         float64              0     0.000000        77   
cb_person_default_on_file       str              0     0.000000         2   
cb_person_cred_hist_length    int64              0     0.000000  